In [0]:
/*
Static lookup tables to represent fiscal quarters.
*/
with dates as (
   select cast(m as date) as m, fq, fy, q, last_day(m) as last_day_of_month from (
    values 
      ('2025-02-01', 'FY26-Q1', 2026, 1),
      ('2025-03-01', 'FY26-Q1', 2026, 1),
      ('2025-04-01', 'FY26-Q1', 2026, 1),
      ('2025-05-01', 'FY26-Q2', 2026, 2),
      ('2025-06-01', 'FY26-Q2', 2026, 2),
      ('2025-07-01', 'FY26-Q2', 2026, 2),
      ('2025-08-01', 'FY26-Q3', 2026, 3),
      ('2025-09-01', 'FY26-Q3', 2026, 3),
      ('2025-10-01', 'FY26-Q3', 2026, 3),
      ('2025-11-01', 'FY26-Q4', 2026, 4),
      ('2025-12-01', 'FY26-Q4', 2026, 4),
      ('2026-01-01', 'FY26-Q4', 2026, 4),
      ('2026-02-01', 'FY27-Q1', 2027, 1),
      ('2026-03-01', 'FY27-Q1', 2027, 1),
      ('2026-04-01', 'FY27-Q1', 2027, 1),
      ('2026-05-01', 'FY27-Q2', 2027, 2),
      ('2026-06-01', 'FY27-Q2', 2027, 2),
      ('2026-07-01', 'FY27-Q2', 2027, 2),
      ('2026-08-01', 'FY27-Q3', 2027, 3),
      ('2026-09-01', 'FY27-Q3', 2027, 3),
      ('2026-10-01', 'FY27-Q3', 2027, 3),
      ('2026-11-01', 'FY27-Q4', 2027, 4),
      ('2026-12-01', 'FY27-Q4', 2027, 4),
      ('2027-01-01', 'FY27-Q4', 2027, 4) 
  ) as dates(m, fq, fy, q)
),

financial_quarters as (
  select fq, fy, q
    , max(last_day_of_month) as last_day_of_quarter    
    , case when getdate() > last_day_of_quarter then q else null end as last_closed_q
    , case when getdate() > last_day_of_quarter then true else false end as is_quarter_closed
    , sum(day(last_day_of_month)) as days_in_quarter
    , (select max(usage_date) from main.gtm_gold.individual_consumption_daily)  as latest_usage_date
    , greatest(0, least(days_in_quarter, datediff(last_day_of_quarter, latest_usage_date))) as days_left_in_quarter
  from dates
  group by fq, fy, q
),

targets as (
  SELECT user_id, Email, dollars, fiscal_year, fiscal_quarter
  FROM gtm_silver.targets_individual
  where Business_Unit = :business_unit
  and Region_Level_1 = :region_level_1
  and Region_Level_2 = :region_level_2
  and type_target = 'dbu_target'
  and Email = :ae_email
  and snapshot_date = (select max(snapshot_date) from gtm_silver.targets_individual where Region_Level_1 = :region_level_1 and Region_Level_2 = :region_level_2)
),

sales_forecast as (
  select f.forecast_owner_id as user_id, d.fy, d.q, f.fiscal_quarter_start_date, f.fiscal_quarter_end_date, f.submitted_my_call, f.submitted_my_call_w_closed_month_actuals
  from main.gtm_data.core_individual_sales_forecast as f
  inner join dates d
  on d.m = f.fiscal_quarter_start_date
  where Business_Unit = :business_unit
  and Region_Level_1 = :region_level_1
  and f.Region_Level_2 = :region_level_2
  --and User_Role_Name = concat('Manager-',:business_unit,'-',:region_level_1,'-',:region_level_2)
  and f.Email = :ae_email
  and f.snapshot_date = (select max(snapshot_date) from main.gtm_data.core_individual_sales_forecast where Region_Level_1 = :region_level_1 and Region_Level_2 = :region_level_2)
),

actuals as (
  select :ae_email as user_id, fy, d.q, c.fiscal_quarter_start_date, sum(c.dbu_dollars_qtd) as dbu_actuals, sum(c.dbu_dollars_t7d_avg) as dbu_dollars_t7d_avg
  from main.gtm_gold.materialized__view_account_obt as c
  left outer join main.gtm_silver.account_dim as b
  on c.account_id = b.account_id
  inner join dates d
  on d.m = c.fiscal_quarter_start_date  
  where b.business_unit = :business_unit
  and b.region_level_1 = :region_level_1
  and b.region_level_2 = :region_level_2
  and c.concatenated_emails like '%' || :ae_email || '%'
  group by all
),

usecases_filtered as (
  select :ae_email as user_id, usecase_id, estimated_monthly_dollar_dbus, target_onboarding_date, target_live_date
    , dateadd(day, 14, date_trunc('month',target_onboarding_date)) as target_onboarding_date_15 
    , dateadd(day, 14, date_trunc('month',target_live_date)) as target_live_date_15 
    , datediff(target_live_date, target_onboarding_date) as total_ramping_days 
    , date_format(dateadd(year, +1, dateadd(month, -1, target_onboarding_date)), "'FY'yy'-Q'Q") as target_onboarding_date_fq
    , date_format(dateadd(year, +1, dateadd(month, -1, target_live_date)), "'FY'yy'-Q'Q") as target_live_date_fq
    , concat('<a href="https://databricks.lightning.force.com/lightning/r/UseCase__c/', usecase_id, '/view" targe="_blank">',usecase_name,'</a>') as usecase_url
    , coalesce(implementation_status, 'Unknown') as implementation_status
    , case 
        when days_in_stage <= 30 or days_in_stage is null then '0-30 days'
        when days_in_stage > 30 and days_in_stage <= 60 then '31-60 days'
        when days_in_stage > 60 and days_in_stage <= 120 then '61-120 days'
        when days_in_stage > 120 then '120+ days'
    end as days_in_stage_bucket
  from gtm_silver.use_case_detail
  where Business_Unit = :business_unit
  and sales_subregion_level_1 = :region_level_1
  and sales_subregion_level_2 = :region_level_2
  and is_incremental = true --Excludes upgrades
  and stage_number <= 5 --Filter out 'Disqualified', 'Lost' and 'Live' UCOs.
  and coalesce(estimated_monthly_dollar_dbus, 0) > 0 -- Excludes zero-valued use cases.
  and concatenated_emails like '%' || :ae_email || '%'
  --and usecase_id = 'aAvVp0000000DN3KAM' 
  /* test cases 
  aAv8Y000000CsLuSAK (Feb25->Sep25), aAvVp000000Uc6IKAS (May25->Sep25), aAv8Y000000lD0ySAE (Jun25->Dec25)
  aAvVp000000W9JVKA0 (Apr25->May25), aAvVp000000d2YEKAY (Feb25->Jul25), aAvVp000000WAqfKAG (Jun25->Jan26)
  */
),

usecases_baselines as (
  select *
    , datediff(getdate(), target_onboarding_date_15) as current_ramping_days  
    -- Calculate this month's baseline for each use case, .i.e. how much are they consuming today? This is used to calculate the actual incremental consumption at the next step.
    -- We assume that the onboarding date and live date occur on day 15 of the month.
    ,case         
        -- if UCO not onboarded yet (i.e. the onboarding date is in the future), then no dbus are generated for the current month.
        when target_onboarding_date_15 > getdate() then 0
        -- if UCO is already live, then it should already realise the expected monthly $DBUs.
        when getdate() > target_live_date_15 then estimated_monthly_dollar_dbus
        -- if UCO is currently onboarding (i.e. the onboarding date is in the past), this is the estimated dbus for the full current month. 
        else round(estimated_monthly_dollar_dbus * try_divide(datediff(getdate(), target_onboarding_date_15), total_ramping_days)) 
      end as current_dbu_baseline 
  from usecases_filtered
),

projections as (
  select uco.user_id, uco.usecase_id, d.fq, d.fy, d.q, d.m, d.last_day_of_month
    , uco.target_onboarding_date, uco.target_onboarding_date_fq, uco.target_live_date, uco.target_live_date_fq, uco.target_onboarding_date_15, uco.target_live_date_15
    , uco.total_ramping_days, uco.current_ramping_days, uco.current_dbu_baseline
    , uco.estimated_monthly_dollar_dbus, uco.implementation_status
    , uco.usecase_url

    --calculate the ramping dbus assuming a linear ramp between the onboarding date and the go-live: from 0 $dbus to the expected monthly $dbus that will be reached on go-live.
    , case       
        when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
        when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus --After the go-live the $dbus remain flat 
        else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) --Between onboarding and go-live the dbus ramp-up linearly
      end as ramping_dbus

    --remove the realised dbus from the ramp, when the use case is ramping up during the onboarding phase, past months' revenue has already been realised.
    , case 
      when d.last_day_of_month < getdate() then 0 --Past month: if a use case started onboarding in the past, and the month is closed then we are removing the consumption from the pipeline to avoid double counting, because we assume it has already been realised (actual dbus).
      when d.last_day_of_month < uco.target_onboarding_date_15 then 0 --Before the onboarding date
      when d.last_day_of_month > uco.target_live_date_15 then uco.estimated_monthly_dollar_dbus - uco.current_dbu_baseline --After the go-live
      else round(uco.estimated_monthly_dollar_dbus * try_divide(datediff(d.last_day_of_month, uco.target_onboarding_date_15), uco.total_ramping_days)) - uco.current_dbu_baseline --Between onboarding and go-live 
    end as ramping_dbus_from_baseline 

  from usecases_baselines as uco
  inner join dates as d --cross join with date table
),

incremental_projections as (
  select *
     -- calculates the actual incremental value substracting last month's $dbus from this month's $dbus.
    , case 
        when getdate() > m then ramping_dbus - ramping_dbus_from_baseline
        else 0 --in the future
    end as dbus_generated
    from projections
),

quarterly_projection as (
  select i.fy, i.fq, i.q, f.last_day_of_quarter, f.last_closed_q, f.days_left_in_quarter, f.is_quarter_closed
    , sum(ramping_dbus) as quarterly_ramping_dbus
    --, max(ramping_dbus) as last_day_of_quarter_dbus
    , sum(dbus_generated) as quarterly_dbus_generated
    -- , sum(case when implementation_status = 'Green' and not is_quarter_closed then dbus_generated else 0 end) as dbus_in_pipeline_green 
    -- , sum(case when implementation_status = 'Yellow' and not is_quarter_closed then dbus_generated else 0 end) as dbus_in_pipeline_yellow 
    -- , sum(case when implementation_status = 'Red' and not is_quarter_closed then dbus_generated else 0 end) as dbus_in_pipeline_Red
    -- , sum(case when implementation_status = 'Unknown' and not is_quarter_closed then dbus_generated else 0 end) as dbus_in_pipeline_unknown
    from incremental_projections as i
    inner join financial_quarters as f
    on i.fq = f.fq
    group by all
),

quarterly_summary as (
  SELECT p.fy, p.fq, p.days_left_in_quarter, p.is_quarter_closed, p.quarterly_dbus_generated
  --, a.dbus_in_pipeline_green, a.dbus_in_pipeline_yellow, a.dbus_in_pipeline_red, a.dbus_in_pipeline_unknown
  , t.dollars as fin_target, f.submitted_my_call, a.dbu_actuals, a.dbu_dollars_t7d_avg
  , coalesce(a.dbu_actuals, 0) as dbu_actuals 
  , case 
      when is_quarter_closed and a.dbu_actuals is null then f.submitted_my_call --If no quarters is closed yet this fiscal year, then use the forecast call.
      when is_quarter_closed and a.dbu_actuals is not null then a.dbu_actuals --use $dbus actuals for all closed quarters. 
      else null
    end as dbu_actuals_closed_quarters

  , case 
      when is_quarter_closed then 0 --closed quarters 
      else dbu_actuals
    end dbu_actuals_current_q

  , case
    when is_quarter_closed then 0
    else dbu_actuals + (dbu_dollars_t7d_avg * days_left_in_quarter * :qoq_organic_growth_multiplier) 
  end as t7d_projection_with_organic_growth

  , p.q - lag(p.last_closed_q) ignore nulls over (order by p.fq asc) as diff

  --, coalesce(lag(dbu_actuals_closed_quarters) ignore nulls over (order by fq asc) * power(:qoq_organic_growth_multiplier, diff), 0) as organic_growth --calculate QoQ organic growth
  --, dbu_actuals_closed_quarters + quarterly_ramping_dbus + organic_growth + quarterly_dbus_generated_current_q as organic_growth_plus_pipeline_all
  --, dbu_actuals_closed_quarters + dbus_in_pipeline_green + organic_growth + quarterly_dbus_generated_current_q as organic_growth_plus_pipeline_green
  
  from quarterly_projection as p
  inner join targets as t 
  on t.fiscal_year = p.fy and t.fiscal_quarter = p.q
  inner join sales_forecast as f 
  on f.fy = p.fy and f.q = p.q
  left outer join actuals as a
  on a.fy = p.fy and a.q = p.q
)
  
select *
from quarterly_summary